# Loading and preprocess

In [ ]:
import os
import time
from pathlib import Path

import anndata as ad
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

from spagrn.regulatory_network import InferNetwork, format_time

# Helper timer (optional)
from contextlib import contextmanager
@contextmanager
def timer(msg: str):
    t0 = time.time()
    print(msg)
    yield
    t1 = time.time()
    print(f"{msg} completed in {format_time(t1 - t0)}\n")
from spagrn.regulatory_network import InferNetwork as irn

In [ ]:
# --------- Configure your inputs here ----------
adata = sc.read_h5ad("Process_Data/bin100_3D_region/combined_adata_subthalamic nucleus.h5ad")
tfs_fn = 'GRN_resource/hs_hgnc_tfs.txt'
database_fn = 'GRN_resource/hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather'
motif_anno_fn = 'GRN_resource/motifs-v10nr_clust-nr.hgnc-m0.001-o0.0.tbl'

niche_human = pd.read_csv('GRN_resource/lr_network_human.csv')
niche_mouse = pd.read_csv('GRN_resource/lr_network_mouse.csv')

output_dir_region = "Output/GRN_output/test_large_function"

# Optional: niche relationships (DataFrame) for Step 6.1
# niches = pd.read_csv("/path/to/niches.csv")
niches = None
receptor_key = "to"

# --------- Parameters aligned to your call ----------
layers = "count"
latent_obsm_key = "align_spatial_3d"
cluster_label = "region_h2"
num_workers = 15
methods = ["FDR_I", "FDR_C", "FDR_G"]
operation = "intersection"
mode = "geary"            # one of: 'moran', 'geary', 'zscore'
model = "danb"
n_neighbors = 10
rho_mask_dropouts = False
local = False
combine = False
somde_k = 20
cache = True
save_tmp = True
weighted_graph = False
umi_counts_obs_key = None
normalize = False


In [ ]:
adata = irn.preprocess(adata, min_genes=10, min_cells=int(len(adata.obs_names)*0.01), min_counts=10, max_gene_num=20000)
X = adata.X
if sp.isspmatrix_csc(X):
    adata.layers['count'] = X.copy()
else:
    if sp.issparse(X):
        adata.layers['count'] = X.tocsc().copy()
    else:
        # dense numpy array -> convert to CSC sparse matrix
        adata.layers['count'] = sp.csc_matrix(X)
del X
print(adata)

In [ ]:
# Pipeline init and sanity checks (mirrors the header in infer)
# ----------------------------------------
print('----------------------------------------')
pipeline_start_time = time.time()

project_name = "split_infer_region"
print(f'Project name is {project_name}')

# Set general output directory
if output_dir_region is None:
    output_dir_region = os.path.dirname(os.path.abspath(__file__))
Path(output_dir_region).mkdir(parents=True, exist_ok=True)
print(f'Saving output files into {output_dir_region}')

# Temporary dir
tmp_dir = os.path.join(output_dir_region, 'tmp_files')
if save_tmp:
    Path(tmp_dir).mkdir(parents=True, exist_ok=True)
    print(f'Saving temporary files to {tmp_dir}')
print('----------------------------------------')

# Sub function test

In [ ]:
# Instantiate InferNetwork
grn = InferNetwork(adata=adata, project_name=project_name)
# Wire tmp_dir used by internal methods
grn.tmp_dir = tmp_dir

# Resolve num_workers / noweights defaults
if num_workers is None:
    from multiprocessing import cpu_count
    num_workers = cpu_count()

noweights = grn.params.get("noweights", False)

exp_mat = grn.data.to_df()

In [ ]:
# %%
# Step 1: Loading TF list...
with timer("Step 1: Loading TF list"):
    if tfs_fn is None:
        # Follow repo behavior (but beware: 'all' may cause issues in geary/moran; prefer providing a TF list)
        tfs = 'all'
        print("Loaded all TFs (tfs='all')")
    else:
        if not os.path.exists(tfs_fn):
            raise FileNotFoundError(f"TF list file not found: {tfs_fn}")
        tfs = grn.load_tfs(tfs_fn)
        print(f"Loaded {len(tfs)} TFs")

# %%
# Step 2: Loading ranking databases...
with timer("Step 2: Loading ranking databases"):
    # Note: load_database uses glob.glob(database_dir)
    dbs = grn.load_database(database_fn)
    try:
        ndb = len(dbs)
    except Exception:
        ndb = "unknown"
    print(f"Loaded {ndb} ranking database(s) from pattern: {database_fn}")

In [ ]:
# %%
# Step 3: Starting GRN Inference (spatial gene co-expression analysis)...
with timer("Step 3: GRN Inference"):
    adjacencies = grn.spg(
        data=grn.data,
        gene_list=None,
        tf_list=tfs,
        jobs=num_workers,
        layer_key=layers,
        model=model,
        latent_obsm_key=latent_obsm_key,
        umi_counts_obs_key=umi_counts_obs_key,
        n_neighbors=n_neighbors,
        weighted_graph=weighted_graph,
        cache=cache,
        save_tmp=save_tmp,
        fn=os.path.join(tmp_dir, f'{mode}_adj.csv'),
        local=local,
        methods=methods,
        operation=operation,
        combine=combine,
        mode=mode,
        somde_k=somde_k,
        fast=False
    )
    print(f"Adjacencies/local correlations shape: {adjacencies.shape if hasattr(adjacencies, 'shape') else 'n/a'}")

In [ ]:
# %%
# Step 4: Computing co-expression modules...
with timer("Step 4: Computing modules"):
    modules = grn.get_modules(
        adjacencies=adjacencies,
        matrix=exp_mat,
        cache=cache,
        save_tmp=save_tmp,
        rho_mask_dropouts=rho_mask_dropouts
    )
    print(f"Modules created: {len(modules)}")


In [ ]:
params = {
            'rank_threshold': 1500,
            'prune_auc_threshold': 0.05,
            'nes_threshold': 3.0,
            'motif_similarity_fdr': 0.05,
            'auc_threshold': 0.05,
            'noweights': False,
        }

print('Step 5: Running regulons prediction (cisTarget)...')
# ctxcore.genesig.Regulon
regulons = grn.prune_modules(modules,
                              dbs,
                              motif_anno_fn,
                              num_workers=num_workers,
                              save_tmp=save_tmp,
                              cache=cache,
                              fn=os.path.join(tmp_dir, 'motifs.csv'),
                              rank_threshold=params["rank_threshold"],
                              auc_threshold=params["prune_auc_threshold"],
                              nes_threshold=params["nes_threshold"],
                              motif_similarity_fdr=params["motif_similarity_fdr"])


In [ ]:
# 6.0. Cellular Enrichment (aka AUCell)
step_start_time = time.time()
print('Step 6.0: Calculating cellular enrichment (AUCell)...')
grn.cal_auc(exp_mat,
             regulons,
             auc_threshold=params["auc_threshold"],
             num_workers=num_workers,
             save_tmp=save_tmp,
             cache=cache,
             noweights=noweights,
             normalize=normalize,
             fn=os.path.join(tmp_dir, 'auc_mtx.csv'))


# 7. Calculate Regulon Specificity Scores
step_start_time = time.time()
print('Step 7: Calculating regulon specificity scores...')
grn.cal_regulon_score(cluster_label=cluster_label, save_tmp=save_tmp,
                       fn=f'{tmp_dir}/regulon_specificity_scores.txt')


In [ ]:
# 8. Save results to h5ad file
print('Step 8: Saving results to h5ad file...')
# dtype=object
output_file = os.path.join(output_dir_region, f'{project_name}_spagrn.h5ad')
grn.data.write_h5ad(output_file)
print(f'Step 8: Results saved to {output_file}')